# CodeAlpha ML Internship — Task 4: Disease Prediction from Medical Data

Predicts disease presence using classification models on the Breast Cancer (Wisconsin) and Heart Disease datasets.

**Author:** Asad Hussain  
**Run this notebook top to bottom in Google Colab (Runtime > Run all).**

In [1]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn xgboost

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                              roc_curve, ConfusionMatrixDisplay)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not installed -> using GradientBoostingClassifier instead.")


def run_pipeline(X, y, dataset_name, target_names):
    print(f"\n{'='*60}\nDataset: {dataset_name}  (n={len(X)}, features={X.shape[1]})\n{'='*60}")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    models = {
        "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        "SVM (RBF)": SVC(probability=True, random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
    }
    if HAS_XGB:
        models["XGBoost"] = XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE)
    else:
        models["Gradient Boosting"] = GradientBoostingClassifier(random_state=RANDOM_STATE)

    results = {}
    plt.figure(figsize=(7, 6))
    for name, model in models.items():
        model.fit(X_train_s, y_train)
        y_pred = model.predict(X_test_s)
        y_proba = model.predict_proba(X_test_s)[:, 1]

        auc = roc_auc_score(y_test, y_proba)
        cv_scores = cross_val_score(model, StandardScaler().fit_transform(X), y, cv=5, scoring="roc_auc")
        results[name] = {
            "Test ROC-AUC": auc,
            "CV ROC-AUC (mean)": cv_scores.mean(),
            "CV ROC-AUC (std)": cv_scores.std(),
        }
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

        print(f"\n--- {name} ---")
        print(classification_report(y_test, y_pred, target_names=target_names))

    plt.plot([0, 1], [0, 1], "--", color="gray")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curves - {dataset_name}")
    plt.legend(); plt.tight_layout()
    fname = f"roc_curves_{dataset_name.replace(' ', '_').lower()}.png"
    plt.savefig(fname, dpi=150)
    plt.close()
    print(f"Saved ROC curves to {fname}")

    results_df = pd.DataFrame(results).T.sort_values("Test ROC-AUC", ascending=False)
    print(f"\n=== {dataset_name}: Model Comparison ===")
    print(results_df.round(3))
    return results_df, models

xgboost not installed -> using GradientBoostingClassifier instead.


## DATASET 1: Breast Cancer Wisconsin (real clinical data, built into sklearn)

In [3]:
bc = load_breast_cancer()
X_bc = pd.DataFrame(bc.data, columns=bc.feature_names)
y_bc = bc.target  # 0 = malignant, 1 = benign
bc_results, bc_models = run_pipeline(X_bc, y_bc, "Breast Cancer", bc.target_names)
bc_results.to_csv("breast_cancer_model_comparison.csv")

# Feature importance for best tree-based model
best_bc_model = bc_models.get("Random Forest")
importances = pd.Series(best_bc_model.feature_importances_, index=X_bc.columns).sort_values(ascending=False)
plt.figure(figsize=(8, 6))
sns.barplot(x=importances.values[:10], y=importances.index[:10])
plt.title("Top 10 Feature Importances - Breast Cancer (Random Forest)")
plt.tight_layout()
plt.savefig("feature_importance_breast_cancer.png", dpi=150)
plt.close()


Dataset: Breast Cancer  (n=569, features=30)

--- Logistic Regression ---
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114




--- SVM (RBF) ---
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114




--- Random Forest ---
              precision    recall  f1-score   support

   malignant       0.93      0.93      0.93        42
      benign       0.96      0.96      0.96        72

    accuracy                           0.95       114
   macro avg       0.94      0.94      0.94       114
weighted avg       0.95      0.95      0.95       114




--- Gradient Boosting ---
              precision    recall  f1-score   support

   malignant       0.97      0.90      0.94        42
      benign       0.95      0.99      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



Saved ROC curves to roc_curves_breast_cancer.png

=== Breast Cancer: Model Comparison ===
                     Test ROC-AUC  CV ROC-AUC (mean)  CV ROC-AUC (std)
Logistic Regression         0.995              0.995             0.004
SVM (RBF)                   0.995              0.996             0.004
Random Forest               0.994              0.992             0.006
Gradient Boosting           0.991              0.991             0.006


## DATASET 2: Heart Disease (UCI) - try OpenML, fallback to synthetic

In [4]:
def load_heart_disease(n_samples=1000):
    try:
        from sklearn.datasets import fetch_openml
        data = fetch_openml("heart-disease", version=1, as_frame=True)
        df = data.frame.copy()
        print("Loaded real UCI Heart Disease dataset from OpenML.")
        target_col = df.columns[-1]
        y = df[target_col].astype(int)
        X = df.drop(columns=[target_col])
        return X, y, ["No Disease", "Disease"]
    except Exception as e:
        print(f"Could not fetch Heart Disease dataset ({e}). Using realistic synthetic data instead.")

    age = np.random.randint(29, 80, n_samples)
    sex = np.random.randint(0, 2, n_samples)
    resting_bp = np.random.normal(130, 17, n_samples).clip(90, 200)
    cholesterol = np.random.normal(240, 45, n_samples).clip(120, 450)
    max_heart_rate = np.random.normal(150, 22, n_samples).clip(70, 210)
    fasting_bs = np.random.binomial(1, 0.15, n_samples)
    exercise_angina = np.random.binomial(1, 0.3, n_samples)
    oldpeak = np.random.exponential(1.0, n_samples).clip(0, 6)

    score = (
        0.03 * age + 0.4 * sex + 0.015 * (resting_bp - 120)
        + 0.01 * (cholesterol - 200) - 0.02 * (max_heart_rate - 150)
        + 1.2 * fasting_bs + 1.0 * exercise_angina + 0.5 * oldpeak
        + np.random.normal(0, 2.5, n_samples)
    )
    disease = (score > np.median(score)).astype(int)

    X = pd.DataFrame({
        "age": age, "sex": sex, "resting_bp": resting_bp.round(1),
        "cholesterol": cholesterol.round(1), "max_heart_rate": max_heart_rate.round(1),
        "fasting_blood_sugar": fasting_bs, "exercise_angina": exercise_angina,
        "oldpeak": oldpeak.round(2)
    })
    return X, disease, ["No Disease", "Disease"]

X_hd, y_hd, hd_target_names = load_heart_disease()
hd_results, hd_models = run_pipeline(X_hd, y_hd, "Heart Disease", hd_target_names)
hd_results.to_csv("heart_disease_model_comparison.csv")

print("\nDone. Task 4 - Disease Prediction complete (Breast Cancer + Heart Disease).")

/usr/local/lib/python3.12/dist-packages/sklearn/datasets/_openml.py:109: UserWarning: A network error occurred while downloading https://api.openml.org/api/v1/json/data/list/data_name/heart-disease/limit/2/data_version/1. Retrying...
  warn(


Could not fetch Heart Disease dataset (HTTP Error 403: Forbidden). Using realistic synthetic data instead.

Dataset: Heart Disease  (n=1000, features=8)

--- Logistic Regression ---
              precision    recall  f1-score   support

  No Disease       0.64      0.70      0.67       100
     Disease       0.67      0.61      0.64       100

    accuracy                           0.66       200
   macro avg       0.66      0.66      0.65       200
weighted avg       0.66      0.66      0.65       200




--- SVM (RBF) ---
              precision    recall  f1-score   support

  No Disease       0.65      0.66      0.65       100
     Disease       0.65      0.64      0.65       100

    accuracy                           0.65       200
   macro avg       0.65      0.65      0.65       200
weighted avg       0.65      0.65      0.65       200




--- Random Forest ---
              precision    recall  f1-score   support

  No Disease       0.60      0.64      0.62       100
     Disease       0.61      0.57      0.59       100

    accuracy                           0.60       200
   macro avg       0.61      0.60      0.60       200
weighted avg       0.61      0.60      0.60       200




--- Gradient Boosting ---
              precision    recall  f1-score   support

  No Disease       0.64      0.64      0.64       100
     Disease       0.64      0.64      0.64       100

    accuracy                           0.64       200
   macro avg       0.64      0.64      0.64       200
weighted avg       0.64      0.64      0.64       200



Saved ROC curves to roc_curves_heart_disease.png

=== Heart Disease: Model Comparison ===
                     Test ROC-AUC  CV ROC-AUC (mean)  CV ROC-AUC (std)
Logistic Regression         0.712              0.701             0.026
SVM (RBF)                   0.693              0.668             0.024
Gradient Boosting           0.647              0.621             0.022
Random Forest               0.645              0.654             0.030

Done. Task 4 - Disease Prediction complete (Breast Cancer + Heart Disease).
